# supply chain decision support system

**Tools:** python,SQL,pandas,Numpy,matplotlib,scipy,sklearn

**Goal:** Analyze and optimize supply chain using operations Research and machine learning

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error
print("All libraries imported !")

All libraries imported !


In [2]:
# DataBase connection
conn=sqlite3.Connection("E:/pythonn/supply_chain_project/supply_chain.db")
cursor =conn.cursor()

#creare tables

cursor.executescript("""
    CREATE TABLE IF NOT EXISTS factories(
        id         INTEGER PRIMARY KEY,
        name       TEXT,
        capacity   INTEGER,
        location   TEXT
    );
    CREATE TABLE IF NOT EXISTS stores(
        id         INTEGER PRIMARY KEY,
        name       TEXT,
        demand     INTEGER,
        location   TEXT
    );
    CREATE TABLE IF NOT EXISTS transport_cost(
        factory_id   INTEGER,
        store_id     INTEGER,
        cost         REAL
    );
     CREATE TABLE IF NOT EXISTS sales_history (
        id       INTEGER PRIMARY KEY,
        store_id INTEGER,
        month    INTEGER,
        sales    INTEGER
    );
""")

conn.commit()
print("Database created !")

Database created !


In [3]:
# Insert data
 # Insert data
cursor.executescript("""
    INSERT OR IGNORE INTO factories VALUES
        (1, 'Factory_North', 100, 'North'),
        (2, 'Factory_South', 80,  'South'),
        (3, 'Factory_East',  120, 'East');
    
    INSERT OR IGNORE INTO stores VALUES
        (1, 'Store_A', 60, 'City_A'),
        (2, 'Store_B', 50, 'City_B'),
        (3, 'Store_C', 70, 'City_C'),
        (4, 'Store_D', 40, 'City_D');
    
    INSERT OR IGNORE INTO transport_cost VALUES
        (1,1,2), (1,2,3), (1,3,1), (1,4,4),
        (2,1,5), (2,2,4), (2,3,8), (2,4,2),
        (3,1,3), (3,2,6), (3,3,2), (3,4,7);
    
    INSERT OR IGNORE INTO sales_history VALUES
        (1,1,1,55), (2,1,2,60), (3,1,3,58), (4,1,4,65),
        (5,1,5,70), (6,1,6,68), (7,1,7,75), (8,1,8,72),
        (9,2,1,45), (10,2,2,48),(11,2,3,50),(12,2,4,52),
        (13,2,5,55),(14,2,6,58),(15,2,7,60),(16,2,8,63),
        (17,3,1,65),(18,3,2,68),(19,3,3,70),(20,3,4,72),
        (21,3,5,75),(22,3,6,78),(23,3,7,80),(24,3,8,82),
        (25,4,1,35),(26,4,2,38),(27,4,3,40),(28,4,4,42),
        (29,4,5,45),(30,4,6,48),(31,4,7,50),(32,4,8,52);
""")

conn.commit()
print(" Data inserted!")

 Data inserted!


In [4]:
# Display all data

df_factories = pd.read_sql("SELECT * FROM factories",conn)
df_stores =    pd.read_sql("SELECT * FROM stores",conn)
df_costs =    pd.read_sql("SELECT * FROM transport_cost",conn)

print("               Factories :      ")
print(df_factories)

print("\n              stores           ")
print(df_stores)

print("\n       Transportation costs    ")
print(df_costs.pivot(index="factory_id", columns="store_id", values="cost"))


               Factories :      
   id           name  capacity location
0   1  Factory_North       100    North
1   2  Factory_South        80    South
2   3   Factory_East       120     East

              stores           
   id     name  demand location
0   1  Store_A      60   City_A
1   2  Store_B      50   City_B
2   3  Store_C      70   City_C
3   4  Store_D      40   City_D

       Transportation costs    
store_id      1    2    3    4
factory_id                    
1           2.0  3.0  1.0  4.0
2           5.0  4.0  8.0  2.0
3           3.0  6.0  2.0  7.0
